# 📋 01. Data Cleaning, Validation & Order-Level Aggregation
**Project**: E-Commerce Discount Impact Analysis & Profit Prediction  
**Role**: Senior Data Scientist & Data Analyst

---

## Overview & Objectives
This notebook handles the foundational data engineering and quality assurance stage:
1. **Load Raw Data**: Ingest the 5 relational tables (`Customers`, `Orders`, `OrderDetails`, `Products`, `Returns`).
2. **Quality & Integrity Audits**: Validate schema, identify nulls, duplicates, and test referential integrity (foreign keys).
3. **Business Logic Verification**: Mathematically verify `SalesAmount` and `Profit` calculations.
4. **Delivered Orders Filtering**: Subset the population to realized transactions (`OrderStatus == 'Delivered'`).
5. **Order-Level Aggregation**: Resolve the line-item vs. order-level unit dilemma to establish the true **Average Order Value (AOV)** analytical unit.
6. **Save Processed Datasets**: Export clean datasets for downstream EDA, statistical testing, and machine learning.


## 🗄️ Relational Database Schema Architecture

![E-commerce Database Schema](reports/figures/E-commerce%20Database%20Schema%20Diagram.png)


In [1]:
import os
from typing import Dict, Tuple
import pandas as pd
import numpy as np

# Verify paths
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
print("Working environment ready.")

Working environment ready.


## Step 1: Ingest Raw Datasets

In [2]:
def load_raw_data(data_dir: str = 'data/raw') -> Dict[str, pd.DataFrame]:
    # Loads all five raw e-commerce CSV files into pandas DataFrames
    return {
        'customers': pd.read_csv(os.path.join(data_dir, 'Customers.csv')),
        'orders': pd.read_csv(os.path.join(data_dir, 'Orders.csv')),
        'order_details': pd.read_csv(os.path.join(data_dir, 'OrderDetails.csv')),
        'products': pd.read_csv(os.path.join(data_dir, 'Products.csv')),
        'returns': pd.read_csv(os.path.join(data_dir, 'Returns.csv')),
    }

datasets = load_raw_data('data/raw')
for name, df in datasets.items():
    print(f"Table: {name:15s} | Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Table: customers       | Shape: 1,000 rows x 7 columns
Table: orders          | Shape: 5,000 rows x 5 columns
Table: order_details   | Shape: 12,558 rows x 8 columns
Table: products        | Shape: 82 rows x 5 columns
Table: returns         | Shape: 257 rows x 4 columns


## Step 2: Relational Integrity & Quality Audits

In [3]:
def validate_data_integrity(datasets: Dict[str, pd.DataFrame]) -> Dict[str, any]:
    # Audits null values, duplicates, foreign key relationships, and formula consistency
    orders = datasets['orders']
    od = datasets['order_details']
    cust = datasets['customers']
    prod = datasets['products']
    ret = datasets['returns']

    audit_results = {
        'shapes': {name: df.shape for name, df in datasets.items()},
        'null_counts': {name: df.isnull().sum().to_dict() for name, df in datasets.items()},
        'duplicate_rows': {name: int(df.duplicated().sum()) for name, df in datasets.items()},
        'orphan_order_details': int((~od['OrderID'].isin(orders['OrderID'])).sum()),
        'orphan_orders_customers': int((~orders['CustomerID'].isin(cust['CustomerID'])).sum()),
        'orphan_order_details_products': int((~od['ProductID'].isin(prod['ProductID'])).sum()),
        'orphan_returns': int((~ret['OrderID'].isin(orders['OrderID'])).sum()),
        'invalid_quantity': int((od['Quantity'] <= 0).sum()),
        'invalid_unit_price': int((od['UnitPrice'] <= 0).sum()),
        'invalid_discount_range': int(((od['DiscountPct'] < 0) | (od['DiscountPct'] > 100)).sum()),
    }

    # Verify SalesAmount formula: UnitPrice * Quantity * (1 - DiscountPct / 100)
    expected_sales = od['UnitPrice'] * od['Quantity'] * (1 - od['DiscountPct'] / 100)
    audit_results['max_sales_discrepancy'] = float((expected_sales - od['SalesAmount']).abs().max())

    # Verify Profit formula: SalesAmount - CostPrice * Quantity
    merged_prod = od.merge(prod[['ProductID', 'CostPrice']], on='ProductID', how='left')
    expected_profit = merged_prod['SalesAmount'] - (merged_prod['CostPrice'] * merged_prod['Quantity'])
    audit_results['max_profit_discrepancy'] = float((expected_profit - merged_prod['Profit']).abs().max())

    return audit_results

audit_report = validate_data_integrity(datasets)
print("=== DATA QUALITY & INTEGRITY REPORT ===")
for k, v in audit_report.items():
    if k != 'null_counts' and k != 'shapes':
        print(f"  {k:32s}: {v}")
print("\nShapes:", audit_report['shapes'])

=== DATA QUALITY & INTEGRITY REPORT ===
  duplicate_rows                  : {'customers': 0, 'orders': 0, 'order_details': 0, 'products': 0, 'returns': 0}
  orphan_order_details            : 0
  orphan_orders_customers         : 0
  orphan_order_details_products   : 0
  orphan_returns                  : 0
  invalid_quantity                : 0
  invalid_unit_price              : 0
  invalid_discount_range          : 0
  max_sales_discrepancy           : 7.275957614183426e-12
  max_profit_discrepancy          : 3.637978807091713e-12

Shapes: {'customers': (1000, 7), 'orders': (5000, 5), 'order_details': (12558, 8), 'products': (82, 5), 'returns': (257, 4)}


## Step 3: Filter Delivered Orders & Aggregate to Order-Level Units

> ⚠️ **Methodological Rule**: An order in this store contains between 1 and 4 line items.  
> Average Order Value (**AOV**) is total checkout expenditure per **Order**, not per line item.  
> We aggregate `OrderDetails` rows by `OrderID` to construct true order-level units.


In [4]:
def clean_and_prepare_delivered_orders(
    datasets: Dict[str, pd.DataFrame]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Filters delivered orders and aggregates line items into order-level analytical units
    orders = datasets['orders'].copy()
    od = datasets['order_details'].copy()
    cust = datasets['customers'].copy()
    prod = datasets['products'].copy()

    # Convert timestamps
    orders['OrderDate'] = pd.to_datetime(orders['OrderDate'])
    cust['SignupDate'] = pd.to_datetime(cust['SignupDate'])

    # Filter to Delivered orders only
    delivered_orders = orders[orders['OrderStatus'] == 'Delivered'].copy()
    delivered_items = od[od['OrderID'].isin(delivered_orders['OrderID'])].copy()

    # Merge line items with product metadata
    delivered_items = delivered_items.merge(prod, on='ProductID', how='left')
    delivered_items = delivered_items.merge(
        delivered_orders[['OrderID', 'OrderDate', 'CustomerID', 'PaymentMethod']],
        on='OrderID',
        how='left'
    )

    # Product category item counts per order
    category_crosstab = pd.crosstab(delivered_items['OrderID'], delivered_items['Category'])

    # Aggregate to Order-Level Analytical Unit
    order_agg = delivered_items.groupby('OrderID').agg(
        order_value=('SalesAmount', 'sum'),
        total_profit=('Profit', 'sum'),
        total_quantity=('Quantity', 'sum'),
        item_count=('OrderDetailID', 'count'),
        avg_discount=('DiscountPct', 'mean'),
        max_discount=('DiscountPct', 'max'),
        has_discount=('DiscountPct', lambda s: bool((s > 0).any())),
        all_discount=('DiscountPct', lambda s: bool((s > 0).all()))
    ).reset_index()

    # Realized gross profit margin %
    order_agg['profit_margin_pct'] = (order_agg['total_profit'] / order_agg['order_value']) * 100

    # Merge customer profile and category counts
    final_orders = delivered_orders.merge(order_agg, on='OrderID', how='inner')
    final_orders = final_orders.merge(cust, on='CustomerID', how='left')
    final_orders = final_orders.merge(category_crosstab, on='OrderID', how='left').fillna(0)

    # Temporal & tenure features
    final_orders['OrderYear'] = final_orders['OrderDate'].dt.year
    final_orders['OrderMonth'] = final_orders['OrderDate'].dt.month
    final_orders['OrderDayOfWeek'] = final_orders['OrderDate'].dt.dayofweek
    final_orders['IsWeekend'] = final_orders['OrderDayOfWeek'].isin([5, 6]).astype(int)
    final_orders['CustomerTenureDays'] = (final_orders['OrderDate'] - final_orders['SignupDate']).dt.days

    return final_orders, delivered_items

delivered_orders, delivered_items = clean_and_prepare_delivered_orders(datasets)

print(f"Delivered Orders (Order-Level AOV Units):  {len(delivered_orders):,} rows")
print(f"Delivered Line Items (Item-Level Units):   {len(delivered_items):,} rows")
print("\nDiscount Segment Distribution in Delivered Orders:")
print(delivered_orders['has_discount'].value_counts())

Delivered Orders (Order-Level AOV Units):  4,386 rows
Delivered Line Items (Item-Level Units):   10,970 rows

Discount Segment Distribution in Delivered Orders:
has_discount
True     4230
False     156
Name: count, dtype: int64


## Step 4: Export Processed Datasets

In [5]:
def save_processed_datasets(
    final_orders: pd.DataFrame,
    delivered_items: pd.DataFrame,
    output_dir: str = 'data/processed'
) -> None:
    # Saves clean order-level and item-level data to data/processed
    os.makedirs(output_dir, exist_ok=True)
    orders_path = os.path.join(output_dir, 'orders_delivered_aggregated.csv')
    items_path = os.path.join(output_dir, 'order_items_delivered.csv')
    
    final_orders.to_csv(orders_path, index=False)
    delivered_items.to_csv(items_path, index=False)
    print(f"Saved: {orders_path} ({os.path.getsize(orders_path):,} bytes)")
    print(f"Saved: {items_path} ({os.path.getsize(items_path):,} bytes)")

save_processed_datasets(delivered_orders, delivered_items)
display(delivered_orders.head(3))

Saved: data/processed\orders_delivered_aggregated.csv (775,069 bytes)
Saved: data/processed\order_items_delivered.csv (1,105,538 bytes)


,OrderID,OrderDate,CustomerID,PaymentMethod,OrderStatus,order_value,total_profit,total_quantity,item_count,avg_discount,...,Beauty,Books,Electronics,Fashion,Home,OrderYear,OrderMonth,OrderDayOfWeek,IsWeekend,CustomerTenureDays
0,O00001,2026-01-19,C0484,UPI,Delivered,49930.00,19182.00,7,2,10.0,...,1,0,1,0,0,2026,1,0,0,716
1,O00002,2026-02-14,C0450,Credit Card,Delivered,25521.25,5916.25,5,2,7.5,...,0,0,0,0,2,2026,2,5,1,1002
2,O00003,2025-11-21,C0523,Credit Card,Delivered,12541.90,4543.90,2,1,5.0,...,0,0,0,1,0,2025,11,4,0,262
